[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_target_selection.ipynb)

# Selecting essential targets in M. tuberculosis

**Orange group · Tuberculosis**

*Mycobacterium tuberculosis* has around 4,000 genes and the group needs a shortlist
small enough to study properly. This notebook starts from a genome-wide experiment
that measured, gene by gene, how much each one has to be switched off before the
bacterium stops growing, and narrows it down to the genes that matter in two strains
of *M. tuberculosis* but not in a harmless relative.

## What you will do

- Load three genome-wide knockdown screens: two strains of *M. tuberculosis* and one
  of *M. smegmatis*, a relative that does not cause disease.
- Read the file's own description of its columns, and keep the genes the screen calls
  essential, treating each screen on its own.
- Keep only the genes the experiment measured confidently, and learn what the
  vulnerability index means.
- Keep what survives in both *M. tuberculosis* strains, then remove the genes that are
  equally essential in *M. smegmatis*.
- Look up the UniProt identifier of every target, rank the shortlist by vulnerability
  and download it.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the three screens

The data comes from [Bosch et al.,
2021](https://doi.org/10.1016/j.cell.2021.06.033), who used a technique called
**CRISPRi** on every gene in the genome. CRISPRi does not delete a gene; it turns its
volume down. By using many different guides, each of which turns the volume down by a
different amount, the experiment can ask a better question than "can the bacterium
live without this gene?". It asks **how much of the gene the bacterium can afford to
lose** before it stops growing.

That difference matters for drug discovery. A drug almost never shuts a protein off
completely. If a bacterium copes fine until a gene is 90% switched off, a drug would
have to be extraordinarily good to do anything. If the bacterium is already in trouble
at 30%, a much more ordinary drug will work.

The authors ran the experiment three times over: in **H37Rv**, the laboratory strain
most TB research uses; in **HN878**, a strain isolated from a patient; and in
***M. smegmatis***, a fast-growing relative that lives in soil and does not cause
disease. All three are in one Excel file, one sheet each.

First, the packages and the file. The Setup cell above put the project folder in place, so the data is already here.

In [ ]:
import os

import pandas as pd
import stylia
from scripts import vulnerability

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()
RANDOM_SEED = 42
DATA = "data/mmc3.xlsx"

if not os.path.exists(DATA):
    raise FileNotFoundError(f"not found: {DATA}. Run the Setup cell again.")
print(f"reading {DATA}")

Now read the three sheets. Each one becomes a table of its own, and they stay separate for most of this notebook.

In [ ]:
screens = {name: vulnerability.load_screen(DATA, name) for name in vulnerability.SCREENS}
pd.DataFrame([{"screen": name, "sheet": vulnerability.SCREENS[name],
               "genes": len(df), "columns": df.shape[1]}
              for name, df in screens.items()])

*M. smegmatis* has a bigger genome than *M. tuberculosis*, which is why its screen
covers more genes. Here are the first rows of the H37Rv screen, showing only the
columns this notebook uses.

In [ ]:
KEY = ["locus_tag", "name", "crispr_ess", "certain", "vi", "vi_lower", "vi_upper"]
screens["H37Rv"][KEY].head()

## 2. What the columns say about each gene

Each row is one gene, described by 28 columns. You do not have to guess what any of
them mean: the file carries its own data dictionary in the `Legend` sheet, written by
the authors. Reading it is a better habit than trusting someone's summary, including
this notebook's.

In [ ]:
legend = pd.read_excel(DATA, sheet_name="Legend", header=None,
                       names=["column", "description"]).dropna()
SHOWN = ["locus_tag", "name", "antibacterial", "tnseq_ess", "crispr_ess", "n_guides",
         "str_span", "certain", "Vulnerability Index", "VI Lower Bound", "VI Upper Bound"]
for _, row in legend.drop_duplicates("column").set_index("column").loc[SHOWN].iterrows():
    print(f"{row.name:>20}  {row['description']}")

A few notes on top of those descriptions:

- **`locus_tag`** is the gene's permanent identifier, like `Rv0667`. Every
  *M. tuberculosis* database uses these, so it is what we join on later. The
  spreadsheet writes them as `RVBD0667`; `load_screen` has already rewritten them into
  the standard form.
- **`name`** is the familiar name, like `rpoB`. Many genes do not have one, and for
  those the name is a copy of the locus tag.
- **`crispr_ess`** and **`tnseq_ess`** are two verdicts on the same question from two
  different experiments, and they do not always agree.
- **`antibacterial`** is filled in for only 18 of the 4,052 genes. We never use it to
  choose anything, which is what makes it a fair check in section 7.

Comparing the two essentiality verdicts is worth doing before trusting either.

In [ ]:
pd.crosstab(screens["H37Rv"]["crispr_ess"], screens["H37Rv"]["tnseq_ess"])

> **Note:** TnSeq breaks genes completely rather than turning them down, and returns
> `Uncertain` or `Unknown` for nearly 200 genes. CRISPRi gives a verdict for every
> one. This notebook follows `crispr_ess`, because it is the call that belongs with
> the vulnerability numbers.

One more thing to notice before filtering. Not every row is a protein-coding gene: the
screen also covers ribosomal RNAs and transfer RNAs, which are made of RNA and never
become proteins. They cannot be drug targets in the sense the group cares about, and
they have no protein identifier, so they will drop out on their own at the very end.
They are easy to spot because their identifier is not an `Rv` number.

In [ ]:
h37 = screens["H37Rv"]
not_genes = h37[~h37["locus_tag"].str.startswith("Rv")]
print(f"{len(not_genes)} of {len(h37):,} rows are not protein-coding genes")
not_genes[["locus_tag", "name", "crispr_ess", "vi"]].head()

## 3. Keep the essential genes, one screen at a time

A target has to be essential: if the bacterium grows perfectly well without the gene,
blocking its protein will not help. So the first filter is `crispr_ess == "Essential"`.

Each screen is filtered on its own, and they are **not** merged yet. The two
*M. tuberculosis* strains were grown in separate experiments, and *M. smegmatis* is a
different species. Merging them now would hide exactly the disagreements we want to
use later.

In [ ]:
counts = pd.DataFrame([{"screen": name, "genes": len(df),
                        "essential": int((df["crispr_ess"] == "Essential").sum())}
                       for name, df in screens.items()])
counts["share"] = (counts["essential"] / counts["genes"]).map("{:.1%}".format)
counts

Around one gene in six is essential in *M. tuberculosis*, and far fewer in
*M. smegmatis*. That is not a mistake. *M. tuberculosis* lives only inside a host and
has lost many of the genes a soil bacterium needs to fend for itself, so a larger
share of the genes it still has are indispensable.

## 4. Keep the genes that were measured confidently

Essential is a yes or no. The experiment also produced a number for each gene, the
**vulnerability index**, and a flag saying whether that number can be trusted.

**What the vulnerability index is.** For each gene the authors fitted a curve
describing how much the bacterium suffers as the gene is turned down further and
further, then added up the damage across the whole range from no knockdown to
complete knockdown. In their words, they "integrated the predicted fitness costs for
sgRNAs spanning the sgRNA strength range (0-1) for each gene", and the total "was
summed into one value, which we refer to as the vulnerability index".

**How to read it.** The damage is measured as a loss, so the numbers are negative:
**the more negative the value, the more vulnerable the gene**. A gene near -16 is
already in trouble when it is only partly switched off. A gene near -1 has to be
almost completely shut down before anything happens. That is precisely the difference
between a target a real drug can work on and one it cannot, because no inhibitor ever
blocks all of its target.

**What `certain` means.** The authors only trusted the curve when the experiment
actually covered a wide range of knockdown strengths for that gene and the fit came
out tight. Where it did not, a vulnerability index is still printed, but its 95% range
is so wide that the number says nothing. Only a minority of genes qualify.

`add_selection_flags` turns each rule into a True/False column instead of removing rows, so we can keep counting what every rule costs.

In [ ]:
flagged = {name: vulnerability.add_selection_flags(df) for name, df in screens.items()}
h37 = flagged["H37Rv"]
print(f"{h37['is_certain'].sum():,} of {len(h37):,} genes were measured confidently "
      f"({h37['is_certain'].mean():.0%})")
pd.crosstab(h37["is_essential"], h37["is_certain"])

Essential genes are far more likely to be measured confidently than the rest, which
makes sense: they are the ones where turning the gene down produced a visible effect
to measure.

Plotting the vulnerability index of the confident and the unreliable genes side by
side shows why the flag cannot be ignored.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for certain, color, label in [(False, nc.gray, "not confident"), (True, nc.orange, "confident")]:
    values = h37.loc[h37["is_certain"] == certain, "vi"]
    ax.hist(values, bins=60, range=(-20, 5), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{label} (n={len(values):,})")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Genes",
             title="Most genes in the screen were not measured confidently")

The unreliable genes are spread all over the scale, including the very vulnerable
end. Some of them may genuinely be vulnerable, but the experiment cannot tell us.

So the selection keeps the genes that are **essential and confidently measured**. We
deliberately do not put a cutoff on the vulnerability index itself: it is a smooth
scale with no natural dividing line, and any threshold would be an invention of ours
rather than a result. It is used to rank the shortlist at the end, so the group can
decide for itself how far down the list to go.

In [ ]:
steps = pd.DataFrame([{"screen": name,
                       "genes": len(df),
                       "essential": int(df["is_essential"].sum()),
                       "and confidently measured": int(df["selected"].sum())}
                      for name, df in flagged.items()])
steps

## 5. Genes that pass in both M. tuberculosis strains

H37Rv is a reference strain that has been maintained in laboratories for over a
century. HN878 was isolated from a patient during an outbreak and belongs to the
W-Beijing family, which is widespread and associated with drug resistance. A target
that only holds up in the reference strain is a risk; one that holds up in a clinical
isolate as well is a safer bet.

So we keep the genes selected in **both**, matching them on their locus tag.

In [ ]:
selected = {name: set(df.loc[df["selected"], "locus_tag"])
            for name, df in flagged.items()}
both = selected["H37Rv"] & selected["HN878"]
print(f"H37Rv {len(selected['H37Rv'])}, HN878 {len(selected['HN878'])}, in both {len(both)}")
print(f"only H37Rv {len(selected['H37Rv'] - selected['HN878'])}, "
      f"only HN878 {len(selected['HN878'] - selected['H37Rv'])}")

Most genes agree across the two strains. Plotting one vulnerability index against
the other shows whether the genes kept in both also rate similarly in both, which is
a different question from whether they were called essential.

In [ ]:
pair = flagged["H37Rv"].merge(flagged["HN878"], on="locus_tag", suffixes=("_h37rv", "_hn878"))
in_both = pair["locus_tag"].isin(both)
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for mask, color, label in [(~in_both, nc.gray, "not selected"), (in_both, nc.orange, "in both")]:
    ax.scatter(pair.loc[mask, "vi_h37rv"], pair.loc[mask, "vi_hn878"],
               color=color, alpha=0.5, s=8, label=f"{label} ({int(mask.sum()):,})")
ax.plot([-18, 3], [-18, 3], color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="H37Rv vulnerability index", ylabel="HN878 vulnerability index",
             title="The same gene in the reference and the clinical strain")

The selected genes sit close to the dashed line of equality, so a gene that is
vulnerable in one strain is generally vulnerable in the other. That is reassuring: it
means the ranking we use at the end is not an accident of which strain was measured.

## 6. Remove what is equally essential in M. smegmatis

*M. smegmatis* is a harmless soil-dwelling cousin of *M. tuberculosis*. If a gene is
essential there too, it is likely a gene every bacterium needs: the machinery that
builds proteins, copies DNA, or makes energy. Those are real vulnerabilities, but they
are not *tuberculosis* vulnerabilities. A drug against one would be likely to hit
harmless bacteria as well, including those living in the patient.

Removing them pushes the shortlist towards what is distinctive about the pathogen,
which is what the group's project plan asks for under selectivity.

There is a practical snag. *M. smegmatis* genes have their own identifiers
(`MSMEG0001` and so on), which say nothing about which *M. tuberculosis* gene they
correspond to. What the two screens do share is the common gene name, so that is what
we match on.

In [ ]:
core = flagged["H37Rv"][flagged["H37Rv"]["locus_tag"].isin(both)].copy()
msmeg_selected = set(flagged["Msmeg"].loc[flagged["Msmeg"]["selected"], "name"])
core["in_msmeg"] = core["name"].isin(msmeg_selected)
print(f"{core['in_msmeg'].sum()} of {len(core)} genes are essential in M. smegmatis too")
core.loc[core["in_msmeg"], ["locus_tag", "name", "vi"]].sort_values("vi").head(10)

The most vulnerable genes removed here are ribosomal proteins (`rplF`, `rplE`,
`rplB`, `rpsC`), the protein export machinery (`secY`) and a transcription factor
(`nusG`), which is exactly the kind of gene the comparison is meant to catch.

Matching on names has a limit worth stating plainly.

In [ ]:
unnamed = core["name"] == core["locus_tag"]
print(f"{unnamed.sum()} of {len(core)} genes have no common name, so they could not be "
      f"compared with M. smegmatis at all")
core.loc[unnamed, ["locus_tag", "name", "crispr_ess", "vi"]].head()

> **Note:** those genes stay in the shortlist because there is no evidence against
> them, not because they passed a test. Genes without a common name are usually the
> ones nobody has studied, which makes them interesting and risky in equal measure.

That leaves the shortlist.

In [ ]:
targets = core[~core["in_msmeg"]].copy()
print(f"{len(core)} genes in both strains -> {len(targets)} after removing "
      f"the M. smegmatis overlap")

## 7. Look at what came out

Before trusting a shortlist it is worth asking two questions of it: where does it sit
in the data it came from, and does it contain the things we already know to be true?

The first is a picture. The selected genes should sit towards the vulnerable end of
the scale compared with everything that was discarded.

In [ ]:
kept = h37["locus_tag"].isin(targets["locus_tag"])
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for mask, color, label in [(~kept, nc.gray, "discarded"), (kept, nc.orange, "selected")]:
    values = h37.loc[mask, "vi"]
    ax.hist(values, bins=60, range=(-20, 5), histtype="stepfilled", alpha=0.6, color=color,
            label=f"{label} (n={len(values):,}, median {values.median():.1f})")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Genes",
             title="Where the shortlist sits in the whole screen")

The second question is the stronger test. The screen labelled 18 genes with the drug
that already targets them. Those drugs work, so a sensible shortlist ought to contain
most of them. We never used that column to choose anything, so it is a genuinely
independent check.

In [ ]:
known = h37[h37["antibacterial"].notna()].copy()
known["in_shortlist"] = known["locus_tag"].isin(targets["locus_tag"])
print(f"{known['in_shortlist'].sum()} of {len(known)} genes hit by a known TB drug "
      f"are in the shortlist")
known[["locus_tag", "name", "antibacterial", "vi", "is_certain",
       "in_shortlist"]].sort_values("vi")

Twelve of the eighteen are there, and which six are missing is worth reading
carefully, because they fall out for two different reasons.

`atpE` (bedaquiline), `ribD` and `23S` were never called essential or confidently
measured, so they went in sections 3 and 4. But `gyrB` and `gyrA` (the
fluoroquinolones) and `rpoB` (rifampicin) are essential, confidently measured and
genuinely vulnerable: they were removed in section 6 for being equally essential in
*M. smegmatis*. DNA gyrase and RNA polymerase really are universal bacterial targets,
and the drugs against them work on many species, so losing them is the selectivity
filter doing its job rather than failing.

Look at what survives instead: `inhA` and `kasA` build mycolic acids, `embA`, `embB`
and `dprE1` build the arabinogalactan layer, `mmpL3` exports mycolic acids. These are
the parts of the cell wall that are distinctive to mycobacteria, which is precisely
what the group is looking for.

Notice also how widely their vulnerability indices vary, from `kasA` at -12.7 to
`folC` at -2.6. Drugs exist against targets all along the scale, which is the best
argument for not imposing a cutoff on this number: it ranks candidates, it does not
divide them into good and bad.

> **Exercise:** `folC`, the target of para-aminosalicylic acid, has a vulnerability
> index of only -2.6. If the group had filtered on this number, that target would have
> been thrown away. How far down the ranking in section 8 would you be willing to
> look, and what else would you want to know about a gene before ruling it out?

## 8. Find the UniProt identifier of every target

The group's deliverable is a list of **proteins**, and everything downstream (looking
for a structure, checking whether anyone has made an inhibitor, assessing
druggability) is keyed on a protein identifier rather than a gene one. The standard is
the **UniProt accession**, a code like `P9WGY9`.

The obvious way to get one is to search UniProt for each gene name in turn, but that
is both slow and unreliable: a name search returns *something* for almost any query,
and that something is often the wrong protein or the right protein in the wrong
strain. The safer approach is to ask UniProt for every protein in the
*M. tuberculosis* proteome in a single request and index it on UniProt's own **ordered
locus name**, the `Rv` number. Then the match is exact by construction rather than
something we have to check afterwards.

In [ ]:
lookup = vulnerability.uniprot_lookup()
print(f"{len(lookup):,} locus tags with a UniProt accession, from one request")
lookup.head(3)

Now attach an accession to each target, and look at whatever fails to match rather than letting it disappear.

In [ ]:
annotated = targets.merge(lookup, on="locus_tag", how="left")
found = annotated["uniprot_ac"].notna()
print(f"{found.sum()} of {len(annotated)} targets have an accession")
annotated.loc[~found, ["locus_tag", "name", "vi"]]

The ones that failed are the ribosomal and transfer RNAs from section 2. They have no
UniProt accession because they are never translated into a protein, so failing to find
one is the correct answer, and they can be dropped.

> **Note:** an empty result is not automatically a mistake. It is worth looking at
> what did not match every time, because the reason can be anything from "this is not
> a protein" to "we joined on the wrong column".

What is left is the group's shortlist, ranked from most to least vulnerable.

In [ ]:
proteins = annotated[found].sort_values("vi")
COLUMNS = ["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi", "vi_lower",
           "vi_upper", "reviewed", "antibacterial"]
proteins = proteins[COLUMNS]
print(f"{len(proteins)} protein targets")
proteins.head()

Finally, save it and download it, so the group has the file even after Colab forgets this session.

In [ ]:
os.makedirs("outputs", exist_ok=True)
final_path = "outputs/mtb_selected_targets.csv"
proteins.to_csv(final_path, index=False)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(final_path)
print(f"{len(proteins)} targets written to {final_path}")

## Summary

- You narrowed 4,052 genes down to the ones the screen calls essential (737) and
  measured confidently (552), kept those that pass in the HN878 clinical strain as
  well (512), and removed the 159 that are equally essential in *M. smegmatis*,
  leaving 353 genes and 348 proteins.
- The vulnerability index says how far a gene has to be turned down before the
  bacterium suffers, and the more negative it is the less inhibition a drug would
  need. It is used here to rank the shortlist, not to cut it: the genes with known TB
  drugs against them are spread right across the scale.
- The confidence flag mattered more than it looks. Only about one gene in seven was
  measured well enough for its vulnerability index to mean anything.
- Most of the 18 genes that already have a TB drug against them came through, which is
  good evidence the selection is sensible. 88 of the shortlist have no common name at
  all, and those are the ones nobody has studied.

**Next:** upload `mtb_selected_targets.csv` to the group's Drive folder so everyone
works from the same list, then go through it together. Which of these proteins has a
known structure, and which of them could a small molecule realistically bind?